# Qwen3-TTS Voice Clone

Clone any voice with a ~10 second reference audio using [Qwen3-TTS](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base).

**Steps:**
1. Open Google Colab (T4 GPU)
2. Install dependencies
3. Upload a reference voice
4. Auto-transcribe it with ASR
5. Configure what you want the cloned voice to say
6. Generate and download

## Block 1 — Open Google Colab with T4 GPU

Before running any code, make sure you're using a GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** under "Hardware accelerator"
3. Click **Save**

Why GPU? The TTS model needs ~4GB VRAM in bfloat16. CPU works but is significantly slower.

In [ ]:
import torch

# Check GPU — warn if not available
if not torch.cuda.is_available():
    print("WARNING: No GPU detected.")
    print("Go to Runtime → Change runtime type → T4 GPU")
    print("The model will run on CPU but will be significantly slower.")
else:
    print("GPU detected:", torch.cuda.get_device_name(0))

## Block 2 — Install dependencies

Install the TTS and ASR packages, plus audio processing libraries. The `-q` flag keeps output minimal.

- `qwen-tts` — voice cloning
- `qwen-asr` — auto-transcribe the reference audio for accurate `ref_text`
- `librosa` — audio loading and resampling for ASR

In [ ]:
!pip install -q "qwen-tts>=0.1.0,<1.0.0" "qwen-asr>=0.1.0,<1.0.0" "librosa>=0.10.0,<1.0.0" "soundfile>=0.12.0,<1.0.0" "transformers>=4.44.0,<5.0.0" "accelerate>=0.30.0,<1.0.0"

## Block 3 — Import modules

Import the required libraries for both ASR and TTS pipelines.

In [ ]:
import soundfile as sf
import librosa
from qwen_tts import Qwen3TTSModel
from qwen_asr import Qwen3ASRModel
from IPython.display import Audio, display
from google.colab import files
import os

# Reuse device from Block 1's GPU check
device = "cuda" if torch.cuda.is_available() else "cpu"

## Block 4 — Upload and play the reference voice

Upload any audio file (.wav, .mp3, etc.) containing the voice you want to clone. The file will play back so you can verify it loaded correctly.

**Tips for best results:**
- Use 3–10 seconds of clean, single-speaker speech
- Avoid background music, noise, or reverb
- The speaker should be speaking clearly at a normal pace

In [ ]:
REFERENCE_VOICE_FILE = "reference_voice.wav"
OUTPUT_VOICE_FILE = "cloned_voice.wav"

print("📁 Please select your reference audio file (.wav, .mp3, etc.)")

uploaded_files = files.upload()

if len(uploaded_files) > 0:
    original_filename = list(uploaded_files.keys())[0]
    os.rename(original_filename, REFERENCE_VOICE_FILE)
    print(f"✅ File saved as '{REFERENCE_VOICE_FILE}'")
    print("📻 Playing uploaded voice:")
    display(Audio(REFERENCE_VOICE_FILE))
else:
    print("❌ No file uploaded!")
    raise SystemExit("No audio file provided")

## Block 5 — Auto-transcribe the reference audio

Use Qwen3-ASR to automatically transcribe what's spoken in the reference audio. This transcript becomes the `ref_text` parameter — the model uses it to align text-to-audio features for speaker embedding extraction.

**Why this matters:** If `ref_text` doesn't match the audio, the speaker embedding degrades and cloning quality drops. Auto-transcription eliminates human transcription error.

The ASR model is unloaded after transcription to free VRAM for the TTS model.

In [ ]:
# Choose ASR model — 0.6B is fast and sufficient for transcription
asr_model_name = "Qwen/Qwen3-ASR-0.6B"

# Choose best dtype
if device == "cuda":
    asr_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    asr_dtype = torch.float32

print(f"Loading ASR model: {asr_model_name}")
asr_model = Qwen3ASRModel.from_pretrained(
    asr_model_name,
    dtype=asr_dtype,
    device_map="auto" if device == "cuda" else None
)

# Transcribe the reference audio
print("🎙️ Transcribing reference audio...")
with torch.inference_mode():
    results = asr_model.transcribe(
        audio=REFERENCE_VOICE_FILE,
        language=None,
        return_time_stamps=False
    )

reference_text = results[0].text
print(f"✅ Transcribed: {reference_text}")

# Unload ASR model to free VRAM for TTS
del asr_model
torch.cuda.empty_cache()
print("🧹 ASR model unloaded, VRAM freed")

## Block 6 — Configure voice cloning parameters

- **reference_text**: Auto-transcribed from your reference audio in Block 5
- **generated_text**: The text you want the cloned voice to speak
- **target_language**: Auto-detected from the generated text (Chinese or English)

In [ ]:
# Text to generate with the cloned voice
generated_text = """
On moonlit night, a small cup filled with warm dreams floated gently across the garden! As I sat by the window, my hands shook lightly. Sun began to rise, but not yet completely bright - it was like when you hold something in one hand and wait for it to grow bigger, slowly, carefully. The moon became round as soon as the first light touched the garden. Now the sun was beginning to turn into warm, golden orange! As day approached, I wanted my cup, but I couldn't be sure if sleep was coming
"""

# Target language — auto-detect from generated_text
def detect_language(text):
    chinese_chars = sum(1 for c in text if "\u4e00" <= c <= "\u9fff")
    return "Chinese" if chinese_chars / len(text) > 0.3 else "English"

target_language = detect_language(generated_text)

print(f"Reference text: {reference_text}")
print(f"Target language: {target_language}")
print(f"Text to generate: {len(generated_text)} characters")

## Block 7 — Load the Qwen3-TTS model

Choose `Qwen/Qwen3-TTS-12Hz-0.6B-Base` (faster, lower VRAM) or `Qwen/Qwen3-TTS-12Hz-1.7B-Base` (higher quality, more VRAM).

- `bfloat16` is used on GPU to save memory
- `device_map="auto"` lets the library place the model on the GPU automatically

In [ ]:
# Switch to "Qwen/Qwen3-TTS-12Hz-0.6B-Base" for faster inference (needs less VRAM)
model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

# Check VRAM before loading — warn if 1.7B model might OOM
if device == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Available VRAM: {vram_gb:.1f} GB")
    if "1.7B" in model_name and vram_gb < 16:
        print("WARNING: 1.7B model may OOM on this GPU. Consider switching to 0.6B.")

# Choose best dtype for GPU
if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

model = Qwen3TTSModel.from_pretrained(
    model_name,
    dtype=dtype,
    device_map="auto" if device == "cuda" else None
)

print("TTS model loaded:", model_name)

## Block 8 — Generate the cloned voice

Run the TTS model to generate speech in the cloned voice. This takes 2–5 minutes depending on text length and GPU.

- `ref_text` — auto-transcribed from your reference audio (Block 5)
- `ref_audio` — the uploaded reference audio file
- `language` — the target language (auto-detected from text)

In [ ]:
import time

print("🎵 Generating voice clone...")
print(f"   Reference audio: {REFERENCE_VOICE_FILE}")
print(f"   Language: {target_language}")

start = time.time()

try:
    wavs, sr = model.generate_voice_clone(
        text=generated_text,
        language=target_language,
        ref_audio=REFERENCE_VOICE_FILE,
        ref_text=reference_text
    )
except Exception as e:
    print(f"Voice cloning failed: {e}")
    print("Try a shorter audio file or switch to the 0.6B model")
    raise

elapsed = time.time() - start
print(f"✅ Voice cloning completed in {elapsed:.1f}s")
print(f"   Sample rate: {sr} Hz")
print(f"   Duration: {len(wavs[0]) / sr:.2f} seconds")

## Block 9 — Save and download the cloned voice

Save the generated audio to a file and download it to your computer.

In [ ]:
# Save audio file
sf.write(OUTPUT_VOICE_FILE, wavs[0], sr)
print(f"💾 Audio saved as {OUTPUT_VOICE_FILE}")

# Play it back
display(Audio(OUTPUT_VOICE_FILE))

# Download to your computer
try:
    files.download(OUTPUT_VOICE_FILE)
    print("✅ Download started!")
except Exception as e:
    print(f"If download doesn't work, go to: Files panel > {OUTPUT_VOICE_FILE}")